# LeetCode 4004 — Minimum Moves to Balance Circular Array II bằng Python

- Link: https://leetcode.com/problems/minimum-moves-to-balance-circular-array-ii/
- Độ khó: **Hard**
- Chủ đề: Đồ thị, Luồng chi phí tối thiểu (Min-Cost Flow), Quy hoạch/Toán trên mảng vòng

> **Bài Premium: đề bài viết theo hiểu biết về bài, nên đối chiếu lại nếu có tài khoản Premium.**
> Skeleton của bài Premium không có sẵn; tên phương thức dưới đây (`minMoves`) lấy theo lời giải tham khảo, cần đối chiếu lại trên LeetCode.

Skeleton (theo tham khảo, Python3):

```python
class Solution:
    def minMoves(self, balance: List[int]) -> int:
```

## 1. Giới thiệu & mục tiêu học

Có `n` người ngồi quanh một vòng tròn, người `i` có số dư `balance[i]` (có thể âm). Mỗi nước đi: một người chuyển **đúng 1 đơn vị** cho hàng xóm trái hoặc phải. Tìm số nước đi ít nhất để **mọi người có số dư không âm**; không thể thì trả `-1`.

Đây là bài **vận chuyển trên đồ thị vòng**: người dư (surplus) gửi hàng cho người thiếu (deficit); mỗi đơn vị đi qua `k` cạnh tốn `k` nước. Cách giải tổng quát và đáng tin nhất là **luồng chi phí nhỏ nhất**, nhưng ta sẽ xây dựng nó từ những ý tưởng cơ bản, rồi kiểm chứng bằng nhiều cách độc lập.

Sau notebook này bạn sẽ:

1. Mô hình hoá bài toán thành **luồng trên cạnh** `f_i` (số đơn vị đi qua cạnh `i → i+1`, có dấu) với ràng buộc "số dư cuối không âm".
2. Viết brute force **BFS trên trạng thái** (kể cả khi người gửi buộc phải đang dương) để kiểm chứng.
3. Viết cách **cắt vòng + DP** trên giá trị luồng, một cách độc lập cho giá trị nhỏ.
4. Học **luồng chi phí nhỏ nhất bằng đường tăng ngắn nhất (SSP)**: dạng từng đơn vị, rồi dạng đẩy theo **nút thắt** (bottleneck) — đáp án nộp.
5. Biết trường hợp đặc biệt `sum = 0` có công thức **trung vị** `O(n log n)` và so sánh các cách bằng test ngẫu nhiên có seed.

## 2. Đề bài & ví dụ

**Phát biểu lại.** Cho mảng vòng `balance` độ dài `n`.

- Một nước đi: người `i` chuyển **đúng 1** đơn vị cho `(i+1) % n` hoặc `(i-1) % n`: `balance[i] -= 1`, `balance[neighbor] += 1`.
- Cần mọi `balance[i] >= 0`. Trả về số nước ít nhất, hoặc `-1` nếu không thể.

**Ví dụ 1:** `[-1,2,-1]` → `2` (người 1 gửi một đơn vị sang trái và một sang phải).

**Ví dụ 2:** `[4,-1,-2]` → `3` (người 0 gửi 1 sang người 1, và 2 sang người 2 qua cạnh vòng).

**Ví dụ 3:** `[-3,-3,5]` → `-1`: tổng `-1 < 0` nên không thể.

**Ràng buộc**

- `1 <= n == balance.length <= 1000`
- `-10^5 <= balance[i] <= 10^5`

**Nhận xét mở đầu:** tổng `S = sum(balance)` bất biến sau mọi nước đi. Nếu `S < 0` thì không thể (mọi số dư không âm kéo theo tổng không âm). Nếu `S >= 0` thì luôn làm được vì các người nối thành vòng.

In [1]:
from typing import List

EXAMPLES = [
    ([-1, 2, -1], 2),
    ([4, -1, -2], 3),
    ([-3, -3, 5], -1),
]
for b, expected in EXAMPLES:
    print(f"balance={b}, tổng={sum(b)} -> kỳ vọng {expected}")

balance=[-1, 2, -1], tổng=0 -> kỳ vọng 2
balance=[4, -1, -2], tổng=1 -> kỳ vọng 3
balance=[-3, -3, 5], tổng=-1 -> kỳ vọng -1


## 3. Kiến thức nền

### 3.1. Chỉ số vòng và phép `%`

Hàng xóm của `i` trong vòng `n` người là `(i + 1) % n` và `(i - 1) % n`. Python cho `(-1) % n == n - 1`, nên `(i - 1) % n` đúng cả khi `i = 0`.

In [2]:
n = 5
for i in range(n):
    print(i, "trái:", (i - 1) % n, "phải:", (i + 1) % n)
print((-1) % 5)

0 trái: 4 phải: 1
1 trái: 0 phải: 2
2 trái: 1 phải: 3
3 trái: 2 phải: 4
4 trái: 3 phải: 0
4


### 3.2. `deque` và BFS (tìm đường ngắn nhất trong đồ thị không trọng số)

`collections.deque` là hàng đợi hai đầu: `append` thêm cuối, `popleft` lấy đầu, đều `O(1)`. BFS lan theo "tầng": tầng `k` là các trạng thái cần đúng `k` nước. Nút đầu tiên thoả điều kiện đích có khoảng cách ngắn nhất.

In [3]:
from collections import deque, defaultdict

def bfs_steps(start, goal, limit=100):
    dist = {start: 0}
    q = deque([start])
    while q:
        x = q.popleft()
        if x == goal:
            return dist[x]
        for y in (x + 1, x * 2):
            if y <= limit and y not in dist:
                dist[y] = dist[x] + 1
                q.append(y)
    return -1

print(bfs_steps(1, 10))   # 1 -> 2 -> 4 -> 5 -> 10 hay 1->2->3->... : 4 nước

4


### 3.3. Luồng (flow), cạnh có sức chứa và chi phí, cạnh ngược (residual)

Một **mạng luồng** có đỉnh, và cạnh `u → v` với `cap` (sức chứa) và `cost` (chi phí mỗi đơn vị). **Luồng** là số đơn vị gửi từ nguồn `s` tới đích `t`, không vượt sức chứa. Với mỗi cạnh ta lưu thêm **cạnh ngược** `v → u` có `cap = 0`, `cost = −cost`: khi ta đã gửi `x` đơn vị qua `u → v`, cạnh ngược có `cap = x`, nghĩa là "có thể huỷ bớt `x` đơn vị đã gửi" với chi phí hoàn lại. Cạnh ngược là điều giúp thuật toán **sửa sai** các quyết định tham lam trước đó.

In [4]:
# Cấu trúc cạnh: [đến, sức_chứa, chi_phí, chỉ_số_cạnh_ngược]
graph = [[] for _ in range(3)]

def add_edge(u, v, cap, cost):
    graph[u].append([v, cap, cost, len(graph[v])])
    graph[v].append([u, 0, -cost, len(graph[u]) - 1])

add_edge(0, 1, 5, 2)
add_edge(1, 2, 3, 1)
print(graph)
# gửi 2 đơn vị qua cạnh 0->1: giảm cap cạnh xuôi, tăng cap cạnh ngược
e = graph[0][0]
e[1] -= 2
graph[e[0]][e[3]][1] += 2
print(graph[0][0], graph[1][0])

[[[1, 5, 2, 0]], [[0, 0, -2, 0], [2, 3, 1, 0]], [[1, 0, -1, 1]]]
[1, 3, 2, 0] [0, 2, -2, 0]


## 4. Ý tưởng nền tảng / trực giác

Hãy hình dung "đơn vị hàng" chạy dọc vòng. Một người dư gửi cho người thiếu; đơn vị đi qua `k` cạnh tốn `k` nước, **bất kể** nó đi theo chiều nào. Tính tay các ví dụ:

- `[-1, 2, -1]`: người 1 dư 2, hai người kề đều thiếu 1 → mỗi người cần 1 đơn vị từ hàng xóm: 2 nước.
- `[4, -1, -2]`: tổng thiếu `3`, người 0 dư `4`. Người 1 và người 2 đều là hàng xóm của người 0 (vòng 3 người) → 1 + 2 = 3 nước.
- `[2, 0, -2, 0]` (n = 4): người 2 thiếu 2, người 0 dư 2, nhưng họ **không kề nhau**. Mỗi đơn vị phải đi qua một người ở giữa (người 1 hoặc người 3): 2 đơn vị × 2 cạnh = 4 nước.

Quan sát then chốt: **mỗi đơn vị "đi" từ một chỗ dư tới một chỗ thiếu, chi phí = khoảng cách vòng**. Nên đáp án = tổng chi phí nhỏ nhất để **ghép** các đơn vị dư với các đơn vị thiếu (một bài vận chuyển). Dư thừa (nếu tổng `> 0`) cứ để nguyên tại chỗ, không tốn nước nào.

Hãy kiểm tra các con số tính tay bằng một brute force thật sự ở mục 5.

## 5. Cách 1 — Brute Force: BFS trên không gian trạng thái

**Ý tưởng.** Trạng thái là tuple số dư. Từ mỗi trạng thái, thử mọi cặp (người `i`, hướng) → trạng thái mới. BFS tầng theo số nước; trạng thái đầu tiên có mọi phần tử `>= 0` cho đáp án. Để không vô hạn, ta giới hạn mỗi phần tử trong `[-B, B]` với `B = max|b| + 3`.

Đề không nói người gửi phải đang dương. Ta cài **hai** phiên bản: `strict=False` (đúng chữ đề) và `strict=True` (chỉ người đang `> 0` mới được gửi) rồi so sánh — nếu bằng nhau thì sự mơ hồ không ảnh hưởng đáp án.

In [5]:
def brute_bfs(balance, strict=False):
    n = len(balance)
    if sum(balance) < 0:
        return -1
    if min(balance) >= 0:
        return 0
    B = max(abs(x) for x in balance) + 3
    start = tuple(balance)
    dist = {start: 0}
    q = deque([start])
    while q:
        s = q.popleft()
        for i in range(n):
            if strict and s[i] <= 0:
                continue
            for j in {(i + 1) % n, (i - 1) % n}:
                if j == i:
                    continue
                t = list(s)
                t[i] -= 1
                t[j] += 1
                t = tuple(t)
                if t in dist or min(t) < -B or max(t) > B:
                    continue
                dist[t] = dist[s] + 1
                if min(t) >= 0:
                    return dist[t]
                q.append(t)
    return -1

for b, expected in EXAMPLES:
    print(b, "->", brute_bfs(b), "| strict:", brute_bfs(b, strict=True), "(kỳ vọng", expected, ")")
print(brute_bfs([2, 0, -2, 0]), brute_bfs([3, 0, 0, -3]))   # 4 và 3

[-1, 2, -1] -> 2 | strict: 2 (kỳ vọng 2 )
[4, -1, -2] -> 3 | strict: 3 (kỳ vọng 3 )
[-3, -3, 5] -> -1 | strict: -1 (kỳ vọng -1 )
4 3


Cell trên in `[2,0,-2,0] → 4` đúng như phân tích tay, còn `[3,0,0,-3]` (người 0 và người 3 **kề nhau** trên vòng) → `3`.

**Độ phức tạp.** Số trạng thái cỡ `(2B+1)^(n-1)`: bùng nổ theo `n` và theo giá trị. Chỉ dùng làm trọng tài cho `n ≤ 5`, giá trị nhỏ. **Nút thắt:** ta đang đếm từng nước đi riêng, trong khi nhiều thứ tự nước đi cho cùng kết quả. Chỉ cần biết **mỗi cạnh có bao nhiêu đơn vị đi qua** (luồng), không cần thứ tự.

## 6. Bước đột phá — mô hình luồng trên cạnh

Gọi `f_i` (số nguyên, có dấu) là **luồng ròng** qua cạnh `i → (i+1) % n`: dương = đi theo chiều kim đồng hồ, âm = ngược lại. Hai đơn vị ngược chiều trên cùng cạnh triệt tiêu nhau nên chỉ cần luồng ròng, và **chi phí = Σ |f_i|**.

Số dư cuối của người `i`: `balance[i] + f_{i-1} − f_i`. Điều kiện của đề:

`f_i ≤ f_{i-1} + balance[i]`   với mọi `i` (chỉ số vòng).

Bài toán trở thành: **tối thiểu hoá `Σ |f_i|` thoả các bất đẳng thức trên.**

Hai hệ quả dùng được ngay:

1. Cộng một hằng `c` vào **mọi** `f_i` không làm đổi `f_i − f_{i-1}`: cả vòng "quay" đều là một **luồng vòng** (circulation). Vì vậy ta có thể **cố định luồng ở một cạnh** `x = f_{n-1}` rồi giải phần còn lại như bài trên đường thẳng.
2. Ở dạng đồ thị: nguồn `s` → người dư (`cap = balance`, chi phí 0); người thiếu → đích `t` (`cap = −balance`, chi phí 0); hai cạnh vô hạn chi phí 1 giữa các hàng xóm. Đáp án = chi phí nhỏ nhất để đẩy đủ `D = tổng thiếu` đơn vị luồng. Đây chính là **min-cost flow**.

Ở các mục sau ta hiện thực hoá cả hai cách nhìn và so chúng với BFS.

## 7. Các cách tiếp theo

- **Cách 2:** cắt vòng + DP trên giá trị luồng (độc lập với đồ thị, cho giá trị nhỏ).
- **Cách 3:** min-cost flow bằng **đường tăng ngắn nhất** từng **1 đơn vị** (SSP, giá trị nhỏ).
- **Cách 4:** SSP đẩy theo **nút thắt** (bottleneck) — đáp án nộp, chạy tốt với `n = 1000`, giá trị `10^5`.
- **Cách phụ:** khi `sum = 0` có công thức trung vị.

### 7.1. Cách 2 — Cắt vòng + DP trên giá trị luồng

**Ý tưởng.** Cố định `x = f_{n-1}` (luồng trên cạnh `n-1 → 0`). Với `i = 0, 1, ..., n-2` ta tính

`g_i(v)` = chi phí nhỏ nhất của `f_0..f_i` với `f_i = v`, thoả các ràng buộc.

Công thức chuyển: `g_i(v) = |v| + min_{u ≥ v − b_i} g_{i-1}(u)` vì ràng buộc `v ≤ u + b_i` ⇔ `u ≥ v − b_i`. Điểm khởi đầu: `g_{-1}` chỉ có `u = x` (chi phí 0). Cuối cùng ràng buộc của người `n-1`: `x ≤ f_{n-2} + b_{n-1}`, chi phí thêm `|x|`.

Mỗi `min_{u ≥ ...}` là một **min hậu tố** (suffix minimum) nên tính nhanh bằng mảng. Miền của `v` là `[-R, R]` với `R = Σ|b|`. Cách này chỉ khả thi khi `R` nhỏ — nhưng nó **không dùng đồ thị** nên là một trọng tài độc lập khác.

In [6]:
def dp_cut_cycle(balance):
    n = len(balance)
    if sum(balance) < 0:
        return -1
    R = sum(abs(x) for x in balance)
    BIG = 10 ** 9
    size = 2 * R + 1                       # chỉ số k ứng với giá trị v = k - R
    best = BIG
    for x in range(-R, R + 1):             # x = f_{n-1}
        g = [BIG] * size
        g[x + R] = 0                       # g_{-1}: f_{-1} = x
        for i in range(n - 1):
            # suffix min: sm[k] = min(g[k:])
            sm = g[:]
            for k in range(size - 2, -1, -1):
                sm[k] = min(sm[k], sm[k + 1])
            ng = [BIG] * size
            for v in range(-R, R + 1):
                lo = v - balance[i]        # cần u >= lo
                if lo > R:
                    continue
                m = sm[max(lo, -R) + R]
                if m < BIG:
                    ng[v + R] = abs(v) + m
            g = ng
        # ràng buộc cuối: x <= f_{n-2} + b_{n-1}  <=>  f_{n-2} >= x - b_{n-1}
        lo = x - balance[n - 1]
        if lo <= R:
            m = min(g[max(lo, -R) + R:])
            if m < BIG:
                best = min(best, abs(x) + m)
    return -1 if best >= BIG else best

for b, expected in EXAMPLES:
    assert dp_cut_cycle(b) == expected, (b, dp_cut_cycle(b))
print("Cách 2 khớp 3 ví dụ")

Cách 2 khớp 3 ví dụ


**Độ phức tạp.** Với `R = Σ|b|`: `(2R+1)` giá trị `x`, mỗi giá trị `n` bước, mỗi bước `O(R)` → `O(n · R²)`. Bộ nhớ `O(R)`. Rất chậm khi giá trị lớn (đề cho `R` tới `10^8`) nhưng cho ta một lời giải độc lập về mô hình `Σ|f_i|`.

### 7.2. Cách 3 — Min-cost flow, đường tăng ngắn nhất từng 1 đơn vị

**Ý tưởng (SSP: Successive Shortest Paths).**

1. Dựng đồ thị: nguồn `s = n`, đích `t = n+1`; cạnh `s → i` (`cap = b_i`, `cost = 0`) cho `b_i > 0`; cạnh `i → t` (`cap = −b_i`, `cost = 0`) cho `b_i < 0`; hai cạnh `i → (i±1)` có `cap = vô hạn`, `cost = 1`.
2. Lặp: tìm **đường ngắn nhất** từ `s` tới `t` trong **đồ thị dư** (cạnh còn `cap > 0`, kể cả cạnh ngược có `cost` âm) — dùng SPFA (Bellman-Ford có hàng đợi), vì cạnh ngược có chi phí âm.
3. Đẩy **1 đơn vị** dọc đường đó, cộng `dist[t]` vào chi phí.
4. Lặp đến khi đủ `D` đơn vị (`D` = tổng thiếu).

Định lý nền: luôn đẩy theo đường ngắn nhất trong đồ thị dư cho ra min-cost flow (không có chu trình âm trong đồ thị dư). Cách này đẩy từng đơn vị nên **số vòng lặp = D**, chậm nếu `D` lớn, nhưng dễ hiểu và dễ kiểm chứng.

In [7]:
INF = 10 ** 18

def build_graph(balance, cost_right=1, cost_left=1):
    n = len(balance)
    graph = [[] for _ in range(n + 2)]
    def add_edge(u, v, cap, cost):
        graph[u].append([v, cap, cost, len(graph[v])])
        graph[v].append([u, 0, -cost, len(graph[u]) - 1])
    source, sink = n, n + 1
    for i in range(n):
        if balance[i] > 0:
            add_edge(source, i, balance[i], 0)
        elif balance[i] < 0:
            add_edge(i, sink, -balance[i], 0)
        add_edge(i, (i + 1) % n, INF, cost_right)
        add_edge(i, (i - 1) % n, INF, cost_left)
    return graph, source, sink

def shortest_path_spfa(graph, source, sink):
    num = len(graph)
    dist = [INF] * num
    pnode = [-1] * num
    pedge = [-1] * num
    inq = [False] * num
    dist[source] = 0
    q = deque([source])
    inq[source] = True
    while q:
        u = q.popleft()
        inq[u] = False
        for idx, (v, cap, cost, _) in enumerate(graph[u]):
            if cap > 0 and dist[v] > dist[u] + cost:
                dist[v] = dist[u] + cost
                pnode[v], pedge[v] = u, idx
                if not inq[v]:
                    inq[v] = True
                    q.append(v)
    return dist, pnode, pedge

def ssp(balance, bottleneck, cost_right=1, cost_left=1):
    """Trả (chi_phí, số_lần_tăng_luồng); chi_phí = -1 nếu không đủ luồng."""
    if sum(balance) < 0:
        return -1, 0
    deficit = sum(-x for x in balance if x < 0)
    if deficit == 0:
        return 0, 0
    graph, s, t = build_graph(balance, cost_right, cost_left)
    flow = cost = rounds = 0
    while flow < deficit:
        dist, pnode, pedge = shortest_path_spfa(graph, s, t)
        if dist[t] >= INF:
            break
        push = deficit - flow
        if not bottleneck:
            push = 1
        v = t
        while v != s:                       # tìm nút thắt của đường
            u = pnode[v]
            push = min(push, graph[u][pedge[v]][1])
            v = u
        v = t
        while v != s:                       # cập nhật cạnh xuôi và ngược
            u = pnode[v]
            e = graph[u][pedge[v]]
            e[1] -= push
            graph[v][e[3]][1] += push
            v = u
        flow += push
        cost += push * dist[t]
        rounds += 1
    return (cost if flow == deficit else -1), rounds

for b, expected in EXAMPLES:
    c, r = ssp(b, bottleneck=False)
    assert c == expected, (b, c)
    print(b, "-> chi phí", c, "sau", r, "lần tăng luồng (từng 1 đơn vị)")

[-1, 2, -1] -> chi phí 2 sau 2 lần tăng luồng (từng 1 đơn vị)
[4, -1, -2] -> chi phí 3 sau 3 lần tăng luồng (từng 1 đơn vị)
[-3, -3, 5] -> chi phí -1 sau 0 lần tăng luồng (từng 1 đơn vị)


**Độ phức tạp.** Số lần tăng `= D` (tổng thiếu, có thể tới `~10^8`), mỗi lần SPFA `O(V·E)` xấu nhất. Chỉ dùng cho `D` nhỏ.

### 7.3. Cách 4 — SSP đẩy theo nút thắt (đáp án nộp)

**Ý tưởng.** Sau khi tìm đường ngắn nhất, thay vì đẩy 1 đơn vị, ta đẩy **cả nút thắt** `min(cap)` dọc đường: đó là lượng lớn nhất không phá vỡ ràng buộc nào trên đường, và đường vẫn ngắn nhất trong suốt quá trình đẩy. Nút thắt là một trong ba loại: cạnh `s → i` (người dư hết hàng), cạnh `j → t` (người thiếu đủ hàng), hoặc cạnh ngược (huỷ toàn bộ luồng qua một cạnh vòng). Mỗi lần tăng làm bão hoà **một** cạnh như vậy, nên số lần tăng chỉ cỡ `O(n)` chứ không phụ thuộc độ lớn của `balance`.

In [8]:
for b, expected in EXAMPLES:
    c, r = ssp(b, bottleneck=True)
    assert c == expected
    print(b, "-> chi phí", c, "sau", r, "lần tăng luồng (theo nút thắt)")

# So sánh số lần tăng luồng khi giá trị lớn: bottleneck không phụ thuộc độ lớn
big = [1000, -400, -600, 0, 0, 0]
print("balance =", big, "-> từng 1 đơn vị:", ssp(big, False), "| theo nút thắt:", ssp(big, True))

[-1, 2, -1] -> chi phí 2 sau 2 lần tăng luồng (theo nút thắt)
[4, -1, -2] -> chi phí 3 sau 2 lần tăng luồng (theo nút thắt)
[-3, -3, 5] -> chi phí -1 sau 0 lần tăng luồng (theo nút thắt)
balance = [1000, -400, -600, 0, 0, 0] -> từng 1 đơn vị: (1600, 1000) | theo nút thắt: (1600, 2)


**Độ phức tạp.** `O(n)` lần tăng (theo phân tích của lời giải tham khảo) × SPFA. Đồ thị là một chu trình với hàng xóm chi phí 1 nên SPFA chạy gần như BFS, trên thực tế mỗi lần tăng `≈ O(n)` → tổng `≈ O(n²)`; **xấu nhất** của SPFA là `O(V·E) = O(n²)` mỗi lần tăng → `O(n³)`. Ta đo thực tế ở mục 11.

### 7.4. Cách phụ — khi `sum = 0`: công thức trung vị

Nếu `sum(balance) = 0` thì số dư cuối phải **đúng bằng 0** (không âm và tổng 0), nên `f_i = f_{i-1} + b_i`, tức `f_i = x + P_i` với `P_i` = tổng tiền tố `b_0 + ... + b_i`. Chi phí `Σ |x + P_i|` nhỏ nhất khi `x = −median(P)`. Đây là bài "chia kẹo trên vòng" kinh điển, `O(n log n)`.

In [9]:
def balanced_by_median(balance):
    assert sum(balance) == 0
    P, run = [], 0
    for x in balance:
        run += x
        P.append(run)
    med = sorted(P)[len(P) // 2]
    return sum(abs(p - med) for p in P)

for b in ([-1, 2, -1, 0], [3, -1, -1, -1], [5, -2, -3], [0, 0, 0]):
    assert sum(b) == 0
    print(b, "-> trung vị:", balanced_by_median(b), "| SSP:", ssp(b, True)[0])

[-1, 2, -1, 0] -> trung vị: 2 | SSP: 2
[3, -1, -1, -1] -> trung vị: 4 | SSP: 4
[5, -2, -3] -> trung vị: 5 | SSP: 5
[0, 0, 0] -> trung vị: 0 | SSP: 0


## 8. TRACE từng bước

Ta in từng lần tăng luồng của SSP (nút thắt) trên `[4, -1, -2]` và một ví dụ phải **huỷ** luồng qua cạnh ngược. Nút `S` = nguồn, `T` = đích.

| cột | ý nghĩa |
|---|---|
| đường | các đỉnh dọc đường ngắn nhất trong đồ thị dư |
| dist | độ dài (tổng chi phí) mỗi đơn vị trên đường |
| đẩy | nút thắt: số đơn vị đẩy lần này |
| chi phí | cộng `đẩy × dist` |

In [10]:
def trace_ssp(balance):
    n = len(balance)
    print("balance =", balance, "tổng =", sum(balance))
    if sum(balance) < 0:
        print("tổng < 0 -> -1"); return -1
    deficit = sum(-x for x in balance if x < 0)
    graph, s, t = build_graph(balance)
    name = lambda v: "S" if v == s else ("T" if v == t else str(v))
    flow = cost = 0
    step = 0
    while flow < deficit:
        dist, pnode, pedge = shortest_path_spfa(graph, s, t)
        path, v, push = [t], t, deficit - flow
        while v != s:
            u = pnode[v]
            push = min(push, graph[u][pedge[v]][1])
            v = u
            path.append(v)
        path.reverse()
        v = t
        while v != s:
            u = pnode[v]
            e = graph[u][pedge[v]]
            e[1] -= push
            graph[v][e[3]][1] += push
            v = u
        flow += push
        cost += push * dist[t]
        step += 1
        print(f"lần {step}: đường {' -> '.join(map(name, path))}, dist={dist[t]}, đẩy {push}, luồng {flow}/{deficit}, chi phí {cost}")
    print("=> đáp án", cost)
    return cost

trace_ssp([4, -1, -2])
print()
trace_ssp([3, 0, -2, 0, -1, 0])

balance = [4, -1, -2] tổng = 1
lần 1: đường S -> 0 -> 1 -> T, dist=1, đẩy 1, luồng 1/3, chi phí 1
lần 2: đường S -> 0 -> 2 -> T, dist=1, đẩy 2, luồng 3/3, chi phí 3
=> đáp án 3

balance = [3, 0, -2, 0, -1, 0] tổng = 0
lần 1: đường S -> 0 -> 1 -> 2 -> T, dist=2, đẩy 2, luồng 2/3, chi phí 4
lần 2: đường S -> 0 -> 5 -> 4 -> T, dist=2, đẩy 1, luồng 3/3, chi phí 6
=> đáp án 6


6

Ở ví dụ đầu: lần 1 đi `S → 0 → 1 → T` (dist 1, đẩy 1, vì người 1 chỉ thiếu 1), lần 2 đi `S → 0 → 2 → T` (cạnh vòng `0 → 2` vì vòng 3 người) đẩy 2: tổng `1 + 2 = 3`. Ví dụ thứ hai có người ở xa nhau nên đường dài hơn và nút thắt thay đổi giữa cạnh nguồn, cạnh đích. Ta cũng thử một trường hợp thật sự cần **cạnh ngược** (đi ngược lại luồng đã đẩy):

In [11]:
# Tìm một ví dụ mà SSP có dùng cạnh ngược (cạnh có cost âm trên đường tăng)
def uses_reverse_edge(balance):
    deficit = sum(-x for x in balance if x < 0)
    graph, s, t = build_graph(balance)
    flow = 0
    while flow < deficit:
        dist, pnode, pedge = shortest_path_spfa(graph, s, t)
        push, v, used_rev = deficit - flow, t, False
        while v != s:
            u = pnode[v]
            e = graph[u][pedge[v]]
            push = min(push, e[1]); used_rev |= e[2] < 0
            v = u
        v = t
        while v != s:
            u = pnode[v]; e = graph[u][pedge[v]]
            e[1] -= push; graph[v][e[3]][1] += push
            v = u
        flow += push
        if used_rev:
            return True
    return False

import random
rng = random.Random(3)
for _ in range(2000):
    n = rng.randint(4, 7)
    b = [rng.randint(-4, 4) for _ in range(n)]
    if sum(b) >= 0 and min(b) < 0 and uses_reverse_edge(b):
        print("ví dụ dùng cạnh ngược:", b)
        trace_ssp(b)
        break

ví dụ dùng cạnh ngược: [0, 1, -2, 1, -2, 1, 1]
balance = [0, 1, -2, 1, -2, 1, 1] tổng = 0
lần 1: đường S -> 1 -> 2 -> T, dist=1, đẩy 1, luồng 1/4, chi phí 1
lần 2: đường S -> 3 -> 4 -> T, dist=1, đẩy 1, luồng 2/4, chi phí 2
lần 3: đường S -> 5 -> 4 -> T, dist=1, đẩy 1, luồng 3/4, chi phí 3
lần 4: đường S -> 6 -> 5 -> 4 -> 3 -> 2 -> T, dist=2, đẩy 1, luồng 4/4, chi phí 5
=> đáp án 5


## 9. Code nộp LeetCode

Ta nộp cách 4 (SSP theo nút thắt), kèm xử lý sớm `tổng < 0` và `tổng thiếu = 0`.

In [12]:
from typing import List
from collections import deque

class Solution:
    def minMoves(self, balance: List[int]) -> int:
        if sum(balance) < 0:
            return -1
        n = len(balance)
        total_deficit = sum(-x for x in balance if x < 0)
        if total_deficit == 0:
            return 0
        INF = 10 ** 18
        source, sink = n, n + 1
        graph = [[] for _ in range(n + 2)]

        def add_edge(u, v, cap, cost):
            graph[u].append([v, cap, cost, len(graph[v])])
            graph[v].append([u, 0, -cost, len(graph[u]) - 1])

        for i in range(n):
            if balance[i] > 0:
                add_edge(source, i, balance[i], 0)
            elif balance[i] < 0:
                add_edge(i, sink, -balance[i], 0)
            add_edge(i, (i + 1) % n, INF, 1)
            add_edge(i, (i - 1) % n, INF, 1)

        flow = cost = 0
        while flow < total_deficit:
            dist = [INF] * (n + 2)
            pnode = [-1] * (n + 2)
            pedge = [-1] * (n + 2)
            inq = [False] * (n + 2)
            dist[source] = 0
            q = deque([source])
            inq[source] = True
            while q:
                u = q.popleft()
                inq[u] = False
                for idx, (v, cap, c, _) in enumerate(graph[u]):
                    if cap > 0 and dist[v] > dist[u] + c:
                        dist[v] = dist[u] + c
                        pnode[v], pedge[v] = u, idx
                        if not inq[v]:
                            inq[v] = True
                            q.append(v)
            if dist[sink] >= INF:
                break
            push = total_deficit - flow
            v = sink
            while v != source:                   # nút thắt của đường
                u = pnode[v]
                push = min(push, graph[u][pedge[v]][1])
                v = u
            v = sink
            while v != source:                   # đẩy luồng, cập nhật cạnh ngược
                u = pnode[v]
                e = graph[u][pedge[v]]
                e[1] -= push
                graph[v][e[3]][1] += push
                v = u
            flow += push
            cost += push * dist[sink]
        return cost if flow == total_deficit else -1

for b, expected in EXAMPLES:
    assert Solution().minMoves(b) == expected
print("Code nộp khớp 3 ví dụ")

Code nộp khớp 3 ví dụ


## 10. Vì sao thuật toán đúng

**(a) Mô hình luồng đúng.** Mỗi dãy nước đi cho một luồng ròng `f` trên các cạnh vòng, thoả `balance_i + f_{i-1} − f_i ≥ 0`, và số nước đi ≥ `Σ|f_i|` (mỗi nước đi đổi luồng ròng một cạnh đúng 1). Ngược lại, từ luồng ròng thoả ràng buộc ta dựng được dãy nước đi đúng `Σ|f_i|` nước: phân rã luồng thành các đường từ nơi dư tới nơi thiếu và đưa từng đơn vị dọc đường (đề không cấm người gửi đang âm; hơn nữa BFS `strict` ở mục 12 cho thấy cũng không cần). Do đó **đáp án = min Σ|f_i|**.

**(b) Tương ứng với min-cost flow.** Ràng buộc "mỗi người có số dư cuối ≥ 0" tương đương: mỗi người thiếu nhận đủ `−b_i` đơn vị, người dư gửi tối đa `b_i`, nên cần đẩy đúng `D = Σ_{b_i<0} (−b_i)` đơn vị từ `s` tới `t` với chi phí đơn vị nhỏ nhất (tổng chi phí cạnh vòng = tổng `Σ|f_i|` sau khi triệt tiêu luồng ngược chiều: min-cost không bao giờ giữ hai chiều cùng lúc vì như vậy là chu trình chi phí dương `2`).

**(c) Khả thi ⇔ tổng ≥ 0.** Nếu `Σb < 0` thì không thể, trả `-1`. Nếu `Σb ≥ 0`, tổng dư ≥ tổng thiếu và các cạnh vòng vô hạn nối mọi người → luồng `D` luôn đẩy được.

**(d) SSP đúng.** Đường tăng ngắn nhất trong đồ thị dư luôn giữ bất biến "không có chu trình âm trong đồ thị dư" (chi phí luồng hiện tại nhỏ nhất với giá trị luồng hiện tại), nên khi đạt luồng `D` ta có min-cost flow. Đẩy theo nút thắt tương đương đẩy lần lượt từng đơn vị trên cùng đường (đường ngắn nhất không đổi khi đẩy chừng nào không có cạnh nào hết dung lượng).

**(e) Điểm dừng.** Nếu SPFA không tới được `t` trước khi đủ `D`, thì không đủ luồng → `-1` (không xảy ra khi `Σb ≥ 0`, nhưng giữ làm phòng thủ).

## 11. Phân tích Time/Space

Gọi `n = len(balance)`, `D` = tổng thiếu, `R = Σ|b|`.

| Cách | Time | Space |
|---|---|---|
| BFS trạng thái | `O((2B+1)^(n-1) · n)` | cùng bậc |
| Cắt vòng + DP | `O(n · R²)` | `O(R)` |
| SSP từng đơn vị | `O(D · V · E)` xấu nhất | `O(n)` |
| SSP theo nút thắt (nộp) | `O(n)` lần tăng × SPFA; thực tế `≈ O(n²)`, xấu nhất `O(n³)` | `O(n)` |
| `sum = 0`, trung vị | `O(n log n)` | `O(n)` |

Đo thời gian thực tế với `n = 1000` và các dạng dữ liệu khác nhau:

In [13]:
import time

def timed(b):
    t0 = time.time()
    r = Solution().minMoves(b)
    return r, round(time.time() - t0, 3)

rng = random.Random(1)
n = 1000
# 1) ngẫu nhiên, tổng >= 0
b = [rng.randint(-10**5, 10**5) for _ in range(n)]
b[0] += max(0, -sum(b))
print("ngẫu nhiên:", timed(b), "| số lần tăng luồng:", ssp(b, True)[1])
# 2) xen kẽ dư lớn / thiếu lớn
b = [10**5 if i % 2 == 0 else -10**5 for i in range(n)]
print("xen kẽ ±1e5:", timed(b))
# 3) một người dư rất lớn, mọi người khác thiếu
b = [-100] * (n - 1) + [100 * (n - 1)]
print("một nguồn lớn:", timed(b))
# 4) nửa đầu dư, nửa sau thiếu (xa nhau nhất)
b = [1000] * (n // 2) + [-1000] * (n // 2)
print("nửa dư, nửa thiếu:", timed(b))
# 5) toàn không âm
print("toàn không âm:", timed([5] * n))

ngẫu nhiên: (765345569, 1.062) | số lần tăng luồng: 1478


xen kẽ ±1e5: (50000000, 0.221)


một nguồn lớn: (25000000, 0.477)


nửa dư, nửa thiếu: (125000000, 0.256)
toàn không âm: (0, 0.0)


## 12. Bộ test

Ví dụ đề, edge case, phản ví dụ cho các cách làm sai, và đối chiếu: BFS (tự do và nghiêm ngặt) ↔ cắt vòng + DP ↔ SSP từng đơn vị ↔ SSP nút thắt ↔ code nộp ↔ trung vị (khi `sum = 0`).

In [14]:
S = Solution()
for b, expected in EXAMPLES:
    for fn in (lambda x: brute_bfs(x), lambda x: brute_bfs(x, True), dp_cut_cycle,
               lambda x: ssp(x, False)[0], lambda x: ssp(x, True)[0], S.minMoves):
        assert fn(b) == expected, (b, expected)
print("Ví dụ đề: mọi cách khớp")

Ví dụ đề: mọi cách khớp


In [15]:
# Edge case
assert S.minMoves([0]) == 0
assert S.minMoves([5]) == 0
assert S.minMoves([-1]) == -1
assert S.minMoves([0, 0, 0]) == 0
assert S.minMoves([3, 4, 5]) == 0                 # đã không âm
assert S.minMoves([-1, 1]) == 1                   # hai người
assert S.minMoves([-2, 3]) == 2
assert S.minMoves([1, -2]) == -1                  # tổng âm
assert S.minMoves([2, 0, -2, 0]) == 4             # hai người đối diện
assert S.minMoves([3, 0, 0, -3]) == 3             # kề nhau qua đường vòng
assert S.minMoves([-10**5, 10**5]) == 10**5
assert S.minMoves([-10**5] * 500 + [10**5] * 500) == S.minMoves([10**5] * 500 + [-10**5] * 500)   # đối xứng vòng
print("Edge case: OK")

Edge case: OK


**Phản ví dụ cho các cách nghĩ sai.** Ba lỗi thường gặp (xem thêm mục 13): quên `tổng < 0`, dùng khoảng cách tuyến tính `|i − j|` thay vì khoảng cách vòng, và tham lam "ghép gần nhất" mà không có cạnh ngược. Ta kiểm tra từng lỗi trong mục 13 bằng BFS/flow.

In [16]:
# 1) Đối chiếu ngẫu nhiên nhỏ với BFS (tự do và nghiêm ngặt)
rng = random.Random(2026)
cnt = 0
for it in range(250):
    n = rng.randint(1, 5)
    b = [rng.randint(-3, 3) for _ in range(n)]
    free = brute_bfs(b)
    assert brute_bfs(b, strict=True) == free, b        # sự mơ hồ về "người gửi phải dương" không ảnh hưởng
    assert dp_cut_cycle(b) == free, (b, dp_cut_cycle(b), free)
    assert ssp(b, False)[0] == free, b
    assert ssp(b, True)[0] == free, b
    assert S.minMoves(b) == free, b
    cnt += 1
print(cnt, "input ngẫu nhiên nhỏ: BFS ↔ DP ↔ SSP ↔ code nộp khớp")

250 input ngẫu nhiên nhỏ: BFS ↔ DP ↔ SSP ↔ code nộp khớp


In [17]:
# 2) Đối chiếu trung bình: DP cắt vòng và SSP (BFS không với tới)
rng = random.Random(77)
for it in range(120):
    n = rng.randint(2, 7)
    b = [rng.randint(-4, 5) for _ in range(n)]
    assert dp_cut_cycle(b) == ssp(b, False)[0] == ssp(b, True)[0] == S.minMoves(b), b
print("120 input trung bình: khớp")

# 3) Khi sum = 0: công thức trung vị
rng = random.Random(9)
for it in range(400):
    n = rng.randint(1, 12)
    b = [rng.randint(-20, 20) for _ in range(n - 1)]
    b.append(-sum(b))
    assert balanced_by_median(b) == S.minMoves(b), b
print("400 input sum=0: trung vị khớp code nộp")

# 4) Input lớn: so SSP từng đơn vị (giá trị vừa) với nút thắt
rng = random.Random(5)
for it in range(25):
    n = rng.randint(8, 40)
    b = [rng.randint(-30, 30) for _ in range(n)]
    b[0] += max(0, -sum(b))
    assert ssp(b, False)[0] == ssp(b, True)[0] == S.minMoves(b)
print("25 input n≤40: SSP từng đơn vị ↔ nút thắt khớp")

120 input trung bình: khớp
400 input sum=0: trung vị khớp code nộp
25 input n≤40: SSP từng đơn vị ↔ nút thắt khớp


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Quên kiểm tra tổng âm

Nếu tổng âm, không bao giờ đủ. Nếu bỏ kiểm tra này, SPFA vẫn chạy nhưng không tới được `t` đủ luồng; code ngây thơ có thể trả giá trị sai (chi phí của luồng đẩy được một phần).

In [18]:
def wrong_no_sum_check(balance):
    n = len(balance)
    deficit = sum(-x for x in balance if x < 0)
    if deficit == 0:
        return 0
    graph, s, t = build_graph(balance)
    flow = cost = 0
    while flow < deficit:
        dist, pnode, pedge = shortest_path_spfa(graph, s, t)
        if dist[t] >= INF:
            break
        push, v = deficit - flow, t
        while v != s:
            u = pnode[v]; push = min(push, graph[u][pedge[v]][1]); v = u
        v = t
        while v != s:
            u = pnode[v]; e = graph[u][pedge[v]]
            e[1] -= push; graph[v][e[3]][1] += push; v = u
        flow += push; cost += push * dist[t]
    return cost                                    # SAI: không kiểm tra flow == deficit

print("sai :", wrong_no_sum_check([-3, -3, 5]))     # chi phí của luồng một phần
print("đúng:", Solution().minMoves([-3, -3, 5]))     # -1

sai : 5
đúng: -1


**Cách đúng:** kiểm tra `sum(balance) < 0 → -1` ngay từ đầu (và/hoặc `flow == total_deficit` ở cuối).

### Lỗi 2 — Dùng khoảng cách tuyến tính `|i − j|` thay vì khoảng cách vòng

Trên vòng, khoảng cách giữa `i` và `j` là `min(|i − j|, n − |i − j|)`: người 0 và người `n-1` là hàng xóm!

In [19]:
def wrong_linear(balance):
    # chạy min-cost flow trên ĐƯỜNG THẲNG (bỏ cạnh nối 0 và n-1)
    n = len(balance)
    if sum(balance) < 0: return -1
    deficit = sum(-x for x in balance if x < 0)
    if deficit == 0: return 0
    graph = [[] for _ in range(n + 2)]
    def add_edge(u, v, cap, cost):
        graph[u].append([v, cap, cost, len(graph[v])])
        graph[v].append([u, 0, -cost, len(graph[u]) - 1])
    for i in range(n):
        if balance[i] > 0: add_edge(n, i, balance[i], 0)
        elif balance[i] < 0: add_edge(i, n + 1, -balance[i], 0)
        if i + 1 < n:
            add_edge(i, i + 1, INF, 1); add_edge(i + 1, i, INF, 1)   # SAI: không có cạnh vòng
    flow = cost = 0
    while flow < deficit:
        dist, pn, pe = shortest_path_spfa(graph, n, n + 1)
        if dist[n + 1] >= INF: break
        push, v = deficit - flow, n + 1
        while v != n:
            u = pn[v]; push = min(push, graph[u][pe[v]][1]); v = u
        v = n + 1
        while v != n:
            u = pn[v]; e = graph[u][pe[v]]; e[1] -= push; graph[v][e[3]][1] += push; v = u
        flow += push; cost += push * dist[n + 1]
    return cost

print("sai :", wrong_linear([3, 0, 0, -3]))          # 9: đi hết chiều dài đường thẳng
print("đúng:", Solution().minMoves([3, 0, 0, -3]))    # 3: đi qua cạnh vòng

sai : 9
đúng: 3


**Cách đúng:** thêm cạnh `(i + 1) % n` và `(i − 1) % n`, để người 0 và người `n−1` kề nhau.

### Lỗi 3 — Tham lam "ghép gần nhất" không đảm bảo tối ưu

Một cách nghĩ hấp dẫn: lần lượt cho mỗi người thiếu lấy hàng từ người dư **gần nhất**. Nhưng người dư gần nhất có thể là người mà người thiếu khác cần hơn. SSP có **cạnh ngược** để sửa những quyết định như vậy. Ta tìm một phản ví dụ bằng tìm kiếm ngẫu nhiên có seed:

In [20]:
def greedy_nearest(balance):
    n = len(balance)
    if sum(balance) < 0: return -1
    b = balance[:]
    dist = lambda i, j: min((i - j) % n, (j - i) % n)
    cost = 0
    for i in range(n):                     # duyệt người thiếu theo chỉ số
        while b[i] < 0:
            j = min((k for k in range(n) if b[k] > 0), key=lambda k: dist(i, k))
            take = min(-b[i], b[j])
            b[i] += take; b[j] -= take
            cost += take * dist(i, j)
    return cost

rng = random.Random(8)
found = None
for _ in range(20000):
    n = rng.randint(4, 8)
    b = [rng.randint(-3, 4) for _ in range(n)]
    if sum(b) >= 0 and greedy_nearest(b) != Solution().minMoves(b):
        found = b; break
assert found is not None
print("balance =", found)
print("tham lam gần nhất:", greedy_nearest(found), "| đúng:", Solution().minMoves(found))

balance = [-1, -2, -3, 4, 0, -1, 4, 4]
tham lam gần nhất: 11 | đúng: 9


**Cách đúng:** dùng min-cost flow (hoặc luồng ròng + tối ưu) thay vì ghép tham lam từng cặp.

### Lỗi 4 — Cài đặt SSP mà quên cạnh ngược

Nếu chỉ lưu cạnh xuôi (không có residual), mỗi lần đẩy là quyết định cuối cùng, và kết quả có thể lớn hơn đáp án thật.

In [21]:
def wrong_no_reverse(balance):
    n = len(balance)
    if sum(balance) < 0: return -1
    deficit = sum(-x for x in balance if x < 0)
    graph, s, t = build_graph(balance)
    for u in range(len(graph)):            # SAI: xoá mọi cạnh ngược (cap 0, cost âm hoặc 0 sinh ra từ add_edge)
        for e in graph[u]:
            if e[2] < 0 or (e[1] == 0):
                e[1] = 0
    flow = cost = 0
    while flow < deficit:
        dist, pn, pe = shortest_path_spfa(graph, s, t)
        if dist[t] >= INF: break
        push, v = deficit - flow, t
        while v != s:
            u = pn[v]; push = min(push, graph[u][pe[v]][1]); v = u
        v = t
        while v != s:
            u = pn[v]; e = graph[u][pe[v]]
            e[1] -= push                    # SAI: không cộng lại vào cạnh ngược
            v = u
        flow += push; cost += push * dist[t]
    return cost if flow == deficit else None

rng = random.Random(4)
for _ in range(20000):
    n = rng.randint(4, 8)
    b = [rng.randint(-3, 4) for _ in range(n)]
    if sum(b) >= 0 and wrong_no_reverse(b) != Solution().minMoves(b):
        print("balance =", b, "| không cạnh ngược:", wrong_no_reverse(b), "| đúng:", Solution().minMoves(b))
        break
else:
    print("(không tìm được phản ví dụ trong 20000 thử)")

balance = [-1, 2, -3, 2, -3, 4, -1, 2] | không cạnh ngược: 10 | đúng: 8


**Cách đúng:** mỗi `add_edge` luôn thêm cạnh ngược `cap = 0, cost = −cost`, và khi đẩy `push` thì **trừ** cạnh xuôi và **cộng** cạnh ngược.

## 14. Mẹo & nhận dạng pattern

- **"Chuyển hàng giữa các nút kề nhau, chi phí = số bước"** → vận chuyển trên đồ thị; hãy nghĩ **luồng ròng trên cạnh** và `Σ|f|`.
- **Mảng vòng** → cố định một cạnh (cắt vòng) để quy về đường thẳng; luồng trên cạnh đó là một tham số. Với `sum = 0` tham số tối ưu là **trung vị** của tổng tiền tố (kỹ thuật "chia kẹo/đổi tiền trên vòng").
- **`n ≤ 1000`, đồ thị thưa**: min-cost flow bằng SSP + SPFA là công cụ chuẩn; luôn kiểm tra điều kiện khả thi (`sum ≥ 0`) trước.
- **Cạnh ngược (residual)** là thứ khiến tham lam "tự sửa sai"; thiếu nó thì không còn tối ưu.
- Đẩy theo **nút thắt** thay vì từng đơn vị làm số vòng lặp không phụ thuộc độ lớn giá trị.
- Với bài Premium không có code mẫu: tự viết brute force theo đúng chữ đề (và thử cả các cách đọc đề mơ hồ) trước khi tin vào mô hình.

## 15. Biến thể

1. **Đường thẳng thay vì vòng** (bỏ cạnh `n-1 ↔ 0`): vẫn là min-cost flow, đồ thị là một đường.
2. **Chi phí hai chiều khác nhau** (đi phải tốn `cr`, đi trái tốn `cl`): chỉ đổi chi phí cạnh.
3. **Sức chứa mỗi cạnh có giới hạn**: đổi `INF` thành giá trị hữu hạn; có thể không khả thi.
4. **Số dư cuối phải đúng bằng một mục tiêu `c_i`**: thay "≥ 0" bằng cân bằng chính xác.

Ta cài biến thể 2 bằng cách truyền `cost_right`, `cost_left` vào `build_graph`, và kiểm chứng bằng **Dijkstra trên không gian trạng thái**.

In [22]:
import heapq

def asym_brute(balance, cr, cl):
    n = len(balance)
    if sum(balance) < 0:
        return -1
    if min(balance) >= 0:
        return 0
    B = max(abs(x) for x in balance) + 3
    start = tuple(balance)
    dist = {start: 0}
    pq = [(0, start)]
    while pq:
        d, s = heapq.heappop(pq)
        if d > dist[s]:
            continue
        if min(s) >= 0:
            return d
        for i in range(n):
            for j, w in (((i + 1) % n, cr), ((i - 1) % n, cl)):
                if j == i:
                    continue
                t = list(s); t[i] -= 1; t[j] += 1; t = tuple(t)
                if min(t) < -B or max(t) > B:
                    continue
                if d + w < dist.get(t, 10 ** 18):
                    dist[t] = d + w
                    heapq.heappush(pq, (d + w, t))
    return -1

def asym_flow(balance, cr, cl):
    return ssp(balance, True, cost_right=cr, cost_left=cl)[0]

print(asym_flow([4, -1, -2], 1, 1), asym_flow([4, -1, -2], 1, 5), asym_flow([4, -1, -2], 5, 1))
rng = random.Random(6)
for _ in range(120):
    n = rng.randint(2, 4)
    b = [rng.randint(-2, 3) for _ in range(n)]
    cr, cl = rng.randint(1, 4), rng.randint(1, 4)
    assert asym_flow(b, cr, cl) == asym_brute(b, cr, cl), (b, cr, cl)
print("Biến thể chi phí hai chiều: khớp Dijkstra trên trạng thái")

3 5 4
Biến thể chi phí hai chiều: khớp Dijkstra trên trạng thái


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. BFS trạng thái | thử mọi nước đi | `O((2B+1)^(n-1))` | cùng bậc | trọng tài, `n ≤ 5` |
| 2. Cắt vòng + DP | tham số `x = f_{n-1}`, DP trên giá trị luồng | `O(n·R²)` | `O(R)` | kiểm chứng mô hình, giá trị nhỏ |
| 3. SSP từng đơn vị | đường tăng ngắn nhất, đẩy 1 | `O(D·V·E)` | `O(n)` | dễ hiểu, `D` nhỏ |
| 4. SSP nút thắt (nộp) | đẩy cả nút thắt | `O(n)` lần tăng × SPFA | `O(n)` | `n = 1000`, giá trị `10^5` |
| Phụ. Trung vị | `x = −median(P)` | `O(n log n)` | `O(n)` | chỉ khi `sum = 0` |

## 17. Bài tập (5 bài, từ dễ đến khó)

**Bài 1 (đoán output).** Hàm `can_balance(b)` trả `True` nếu `sum(b) >= 0`. `can_balance([4, -1, -2])`, `can_balance([-3, -3, 5])` là gì? Tổng thiếu `D` của `[4, -1, -2]` bằng bao nhiêu?

**Bài 2 (tính tay).** Tính đáp án cho `[3, 0, 0, -3]` và `[2, 0, -2, 0]` bằng cách đếm số cạnh mỗi đơn vị đi qua.

**Bài 3 (điền TODO).** Hoàn thiện `add_edge_todo(graph, u, v, cap, cost)` để thêm cạnh xuôi và cạnh ngược.

**Bài 4 (tự viết lại).** Viết `balanced_moves(b)` bằng công thức trung vị khi `sum(b) == 0`.

**Bài 5 (biến thể / mini challenge).** Cho thêm số nguyên `K`: mỗi cạnh vòng chỉ cho phép tối đa `K` đơn vị đi qua theo mỗi chiều. Viết `min_moves_capped(b, K)` trả về số nước ít nhất hoặc `-1` nếu không khả thi, và kiểm chứng bằng BFS trên trạng thái với giới hạn đó (không cần quá nhanh).

In [23]:
# Bài 3 — TODO (không gọi khi chạy notebook)
def add_edge_todo(graph, u, v, cap, cost):
    # TODO 1: thêm [v, cap, cost, len(graph[v])] vào graph[u]
    # TODO 2: thêm [u, 0, -cost, len(graph[u]) - 1] vào graph[v]
    pass

# Bài 4 — TODO
def balanced_moves(b):
    # TODO: P = tổng tiền tố; med = trung vị của P; trả sum(|p - med|)
    pass

# Bài 5 — TODO
def min_moves_capped(b, K):
    # TODO: như SSP nhưng cạnh vòng có cap = K thay vì INF; chú ý không khả thi -> -1
    pass

## 18. Gợi ý 3 mức

**Bài 1.**
- Mức 1: chỉ cần so sánh tổng với 0.
- Mức 2: `[4,-1,-2]` tổng `1`, `[-3,-3,5]` tổng `-1`.
- Mức 3: `True`, `False`, và `D = 1 + 2 = 3`.

**Bài 2.**
- Mức 1: kiểm tra hai người có kề nhau trên vòng không.
- Mức 2: `[3,0,0,-3]`: người 0 và 3 kề → mỗi đơn vị đi 1 cạnh → 3. `[2,0,-2,0]`: người 0 và 2 đối diện → mỗi đơn vị đi 2 cạnh → 4.
- Mức 3: đáp án `3` và `4`.

**Bài 3.**
- Mức 1: `graph[u]` và `graph[v]` là danh sách cạnh.
- Mức 2: chỉ số cạnh ngược của cạnh xuôi là `len(graph[v])` **trước** khi thêm.
- Mức 3: `graph[u].append([v, cap, cost, len(graph[v])])` rồi `graph[v].append([u, 0, -cost, len(graph[u]) - 1])`.

**Bài 4.**
- Mức 1: `P[i] = b[0] + ... + b[i]`.
- Mức 2: chọn `x` để `Σ|x + P_i|` nhỏ nhất → `x = −median(P)`.
- Mức 3: `sorted(P)[len(P)//2]` là một trung vị hợp lệ.

**Bài 5.**
- Mức 1: đổi `INF` thành `K` ở hai cạnh vòng.
- Mức 2: có thể không đủ luồng → `flow < deficit` thì trả `-1`.
- Mức 3: BFS trọng tài khó hơn: trạng thái phải nhớ đã dùng bao nhiêu trên mỗi cạnh; thay vào đó đối chiếu bằng `dp_cut_cycle` sửa miền `v ∈ [-K, K]`.

## 19. Đáp án tham khảo

In [24]:
# Bài 1
def can_balance(b):
    return sum(b) >= 0
assert can_balance([4, -1, -2]) and not can_balance([-3, -3, 5])
assert sum(-x for x in [4, -1, -2] if x < 0) == 3

# Bài 2
assert Solution().minMoves([3, 0, 0, -3]) == 3
assert Solution().minMoves([2, 0, -2, 0]) == 4

# Bài 3
def add_edge_todo_ans(graph, u, v, cap, cost):
    graph[u].append([v, cap, cost, len(graph[v])])
    graph[v].append([u, 0, -cost, len(graph[u]) - 1])

g1 = [[] for _ in range(3)]
add_edge_todo_ans(g1, 0, 1, 5, 2)
add_edge_todo_ans(g1, 1, 2, 3, 1)
assert g1 == [[[1, 5, 2, 0]], [[0, 0, -2, 0], [2, 3, 1, 0]], [[1, 0, -1, 1]]]

# Bài 4
def balanced_moves_ans(b):
    assert sum(b) == 0
    P, run = [], 0
    for x in b:
        run += x
        P.append(run)
    med = sorted(P)[len(P) // 2]
    return sum(abs(p - med) for p in P)

rng = random.Random(14)
for _ in range(300):
    n = rng.randint(1, 10)
    b = [rng.randint(-9, 9) for _ in range(n - 1)]
    b.append(-sum(b))
    assert balanced_moves_ans(b) == Solution().minMoves(b)
print("Bài 1–4 OK")

Bài 1–4 OK


In [25]:
# Bài 5: cạnh vòng có sức chứa K theo mỗi chiều
def min_moves_capped_ans(b, K):
    n = len(b)
    if sum(b) < 0:
        return -1
    deficit = sum(-x for x in b if x < 0)
    if deficit == 0:
        return 0
    graph = [[] for _ in range(n + 2)]
    def add_edge(u, v, cap, cost):
        graph[u].append([v, cap, cost, len(graph[v])])
        graph[v].append([u, 0, -cost, len(graph[u]) - 1])
    s, t = n, n + 1
    for i in range(n):
        if b[i] > 0: add_edge(s, i, b[i], 0)
        elif b[i] < 0: add_edge(i, t, -b[i], 0)
        j = (i + 1) % n
        if j != i:
            add_edge(i, j, K, 1)
            add_edge(j, i, K, 1)       # chiều ngược lại cũng sức chứa K
    flow = cost = 0
    while flow < deficit:
        dist, pn, pe = shortest_path_spfa(graph, s, t)
        if dist[t] >= INF:
            break
        push, v = deficit - flow, t
        while v != s:
            u = pn[v]; push = min(push, graph[u][pe[v]][1]); v = u
        v = t
        while v != s:
            u = pn[v]; e = graph[u][pe[v]]
            e[1] -= push; graph[v][e[3]][1] += push; v = u
        flow += push; cost += push * dist[t]
    return cost if flow == deficit else -1

# Trọng tài: DP cắt vòng, nhưng luồng ròng mỗi cạnh bị giới hạn |f_i| <= K
def capped_dp(balance, K):
    n = len(balance)
    if sum(balance) < 0:
        return -1
    if n == 1:
        return 0 if balance[0] >= 0 else -1
    BIG = 10 ** 9
    R = K
    size = 2 * R + 1
    best = BIG
    for x in range(-K, K + 1):
        g = [BIG] * size
        g[x + R] = 0
        for i in range(n - 1):
            sm = g[:]
            for k in range(size - 2, -1, -1):
                sm[k] = min(sm[k], sm[k + 1])
            ng = [BIG] * size
            for v in range(-R, R + 1):
                lo = v - balance[i]
                if lo > R:
                    continue
                m = sm[max(lo, -R) + R]
                if m < BIG:
                    ng[v + R] = abs(v) + m
            g = ng
        lo = x - balance[n - 1]
        if lo <= R:
            m = min(g[max(lo, -R) + R:])
            if m < BIG:
                best = min(best, abs(x) + m)
    return -1 if best >= BIG else best

rng = random.Random(23)
for _ in range(150):
    n = rng.randint(3, 6)       # n >= 3 để hai cạnh i->j và j->i là hai cạnh phân biệt trên vòng
    b = [rng.randint(-3, 3) for _ in range(n)]
    K = rng.randint(1, 4)
    assert min_moves_capped_ans(b, K) == capped_dp(b, K), (b, K, min_moves_capped_ans(b, K), capped_dp(b, K))
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template min-cost flow (SSP + SPFA, đẩy nút thắt):**

```python
if sum(balance) < 0: return -1
D = sum(-x for x in balance if x < 0)
if D == 0: return 0
# đồ thị: s -> người dư (cap = b, cost 0); người thiếu -> t (cap = -b, cost 0)
#         i <-> (i±1) % n  (cap = INF, cost 1); mỗi cạnh có cạnh ngược (cap 0, cost -c)
flow = cost = 0
while flow < D:
    dist, pnode, pedge = spfa(s)            # đường ngắn nhất trong đồ thị dư
    if dist[t] == INF: break
    push = min(D - flow, bottleneck(path))  # nút thắt
    update(path, push)                      # trừ cạnh xuôi, cộng cạnh ngược
    flow += push; cost += push * dist[t]
return cost if flow == D else -1
```

**Khi `sum == 0`:** đáp án = `Σ |P_i − median(P)|` với `P` là tổng tiền tố.

**Checklist trước khi submit:**

- [ ] `sum < 0` → `-1`; không có người thiếu → `0`.
- [ ] Hai cạnh vòng `(i+1) % n` và `(i-1) % n`, cap vô hạn, cost 1.
- [ ] Mỗi cạnh có cạnh ngược (`cap = 0`, `cost = −cost`) và cập nhật cả hai khi đẩy.
- [ ] Đẩy theo nút thắt, tổng chi phí cộng `push × dist[t]`.
- [ ] Đã đối chiếu BFS trạng thái trên input rất nhỏ; đã đối chiếu tên phương thức nếu có tài khoản Premium.